In [1]:
import os
import json
import time
import pandas as pd
from tqdm import tqdm
from openai import OpenAI


In [1]:
import os
import json
import time
import pandas as pd
from tqdm import tqdm
from openai import OpenAI


# ===== Cell 2 =====
API_KEY = ''
client = OpenAI(
    api_key=API_KEY
)  # OPENAI_API_KEY 환경변수에 세팅되어 있어야 함
MODEL_NAME = "gpt-4.1-mini"  # 필요시 변경

DATA_DIR = r"C:\Users\User\data_study\국회회의록pjt\data"


# ===== Cell 3 =====
SYSTEM_PROMPT = """
너는 대한민국 국회 회의록에서
‘실제 자료·문서·파일 제출 요구’만을 정밀하게 추출하는 전문가다.

━━━━━━━━━━━━━━━━━━━━━━
[자료 제출 요구의 엄격한 정의]
━━━━━━━━━━━━━━━━━━━━━━

다음 조건을 모두 만족하는 경우만 ‘자료 제출 요구’로 판단한다.

1. 요구 대상이 명확히 존재해야 한다.
   - 정부, 부처, 청, 위원회, 공공기관, 산하기관 등

2. 요구 내용이 ‘구체적인 자료의 제출’이어야 한다.
   포함 예시:
   - 통계, 현황표, 명단, 보고서, 문서, 자료집
   - 서면 자료, 파일, 내부 문건, 회의자료
   - 조사 결과를 정리한 보고서 형태의 자료

3. 단순 행동·조치·답변 요구는 제외한다.
   반드시 ‘자료를 제출하라 / 내라 / 제공하라’는 의미가 있어야 한다.

━━━━━━━━━━━━━━━━━━━━━━
[반드시 제외할 것]
━━━━━━━━━━━━━━━━━━━━━━

다음은 절대 포함하지 않는다.

- “조치하라”, “개선하라”, “검토하라” 등 행동 요구
- “설명해 달라”, “답변해 달라” 등 구두·서면 답변 요구
- 의견 표명, 문제 제기, 비판, 평가
- 자료 제출 없이 결과만 묻는 질문
- 이미 제출된 자료를 단순 언급하는 발언
- 과거 자료 요구를 회상만 하는 경우

━━━━━━━━━━━━━━━━━━━━━━
[요약 방식 — 매우 중요]
━━━━━━━━━━━━━━━━━━━━━━

‘requested_data’는 반드시 다음 형식을 따른다.

- 어떤 자료를
- 왜 요구하는지(문제의식·검증 목적·확인 목적 등)

을 **하나의 완결된 문장**으로 요약한다.

❌ 나쁜 예:
- “경찰 인력 현황 자료 제출 요구”

✅ 좋은 예:
- “지역별 경찰 인력 배치가 적정한지 검증하기 위해 최근 5년간 인력 현황 자료 제출 요구”

━━━━━━━━━━━━━━━━━━━━━━
[출력 형식]
━━━━━━━━━━━━━━━━━━━━━━

출력은 반드시 JSON 배열만 사용한다.
JSON 외의 다른 텍스트는 절대 출력하지 마라.

[
  {
    "target": "자료 제출을 요구받은 대상(기관/부처/조직)",
    "requested_data": "어떤 자료를 왜 요구하는지 한 문장 요약",
    "category": "아래 범주 중 하나"
  }
]

━━━━━━━━━━━━━━━━━━━━━━
[category 범주]
━━━━━━━━━━━━━━━━━━━━━━
- 범죄·치안
- 재난·안전
- 인사·조직
- 예산·재정
- 정책·제도
- 통계·현황
- 감사·점검
- 기타

━━━━━━━━━━━━━━━━━━━━━━
[출력 규칙]
━━━━━━━━━━━━━━━━━━━━━━

- 하나의 발언에서 여러 자료 제출 요구가 있으면 각각 분리한다.
- 자료 제출 요구가 전혀 없으면 반드시 빈 배열 [] 만 출력한다.
- 판단이 애매한 경우에는 포함하지 말고 제외한다.
"""


# ===== Cell 4 =====
def build_user_prompt(speech_text: str) -> str:
    return f"""
다음은 국회 회의록 발언이다.
위 발언에서 '실제 자료 제출 요구'만 JSON 배열로 추출하라.

발언:
\"\"\"
{speech_text}
\"\"\"
"""


# ===== Cell 5 =====
def extract_requests_from_speech(speech_text: str, model: str = "gpt-4.1-mini"):
    """
    GPT로 실제 자료 제출 요구를 JSON 배열로 추출.
    - quota/429 등 에러 시에도 파이프라인이 중단되지 않도록 [] 반환
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_user_prompt(speech_text)}
            ],
            temperature=0
        )
        content = response.choices[0].message.content.strip()
        return json.loads(content)

    except Exception as e:
        print("GPT 파싱 에러:", e)
        return []


# ===== Cell 6 =====
REQUEST_KEYWORDS = [
    "자료", "제출", "현황", "통계", "보고서", "명단", "내역", "서면", "제공", "주시기 바랍니다", "바랍니다"
]

def is_candidate_request(text: str) -> bool:
    if not isinstance(text, str):
        return False
    t = text.strip()
    if not t:
        return False
    return any(k in t for k in REQUEST_KEYWORDS)


# ===== Cell 7 =====
def build_requester_party_map(df: pd.DataFrame) -> dict:
    """
    ✅ 핵심: 요구자(의원/위원) 정당 매핑 생성
    - df 내 speaker_name별로 party를 매핑
    - party가 '미분류'면 공백으로 처리
    - 보통 요구자료는 의원/위원이 요구하므로,
      speaker_position에 '의원/위원/위원장/간사' 등이 포함된 경우에만 정당을 채움
      그 외(정부/기관/증인 등)는 공백으로 둠
    """
    required_cols = ["speaker_name", "speaker_position", "party"]
    for c in required_cols:
        if c not in df.columns:
            return {}

    tmp = df[required_cols].fillna("")
    tmp["speaker_name"] = tmp["speaker_name"].astype(str).str.strip()
    tmp["speaker_position"] = tmp["speaker_position"].astype(str).str.strip()
    tmp["party"] = tmp["party"].astype(str).str.strip()

    party_map = {}

    for name, g in tmp.groupby("speaker_name"):
        if not name:
            continue

        pos_join = " ".join(g["speaker_position"].tolist())
        is_lawmaker_like = any(k in pos_join for k in ["의원", "위원", "위원장", "간사"])

        parties = [p for p in g["party"].tolist() if p and p != "미분류"]

        if is_lawmaker_like and parties:
            party_map[name] = parties[0]
        else:
            party_map[name] = ""

    return party_map


def get_requester_party(party_map: dict, requester_name: str) -> str:
    return party_map.get(str(requester_name).strip(), "")


# ===== Cell 8 =====
def extract_meeting_round_from_folder(folder_name: str) -> str:
    # '제421회' -> '421회'
    return folder_name.replace("제", "").replace("회", "") + "회"


# ===== Cell 9 =====
def run_request_extraction_pipeline(
    data_dir: str,
    model: str = "gpt-4.1-mini",
    sleep_sec: float = 0.25
) -> pd.DataFrame:
    """
    data_dir 하위의 '제379회'~ 폴더들을 순회하며,
    회차별로 발언자(speaker_name) 발언을 합쳐 '실제 자료 제출 요구'만 추출.

    추가 컬럼:
      - 요구자정당: df 내 party를 speaker_name 기준으로 매핑
                 (단, 의원/위원일 때만 채우고 '미분류'는 공백)

    출력 컬럼:
      회의회차, 요구자명, 요구자정당, 대상, 실제요구자료, 카테고리
    """
    all_results = []
    candidate_count = 0

    folders = [f for f in os.listdir(data_dir) if f.startswith("제") and f.endswith("회")]
    folders.sort(key=lambda x: int(x.replace("제", "").replace("회", "")))

    for folder in folders:
        meeting_round = extract_meeting_round_from_folder(folder)
        folder_path = os.path.join(data_dir, folder)
        print(f"\n▶ 처리 중: {meeting_round}")

        files = [
            fn for fn in os.listdir(folder_path)
            if fn.endswith("_minutes_speeches.csv") or fn.endswith("_minutes_speeches.xlsx")
        ]
        files.sort()
        
        for fn in files:
            file_path = os.path.join(folder_path, fn)
        
            # 🔥 확장자별로 다르게 읽기
            if fn.endswith(".csv"):
                try:
                    df = pd.read_csv(file_path, encoding="utf-8-sig")
                except UnicodeDecodeError:
                    df = pd.read_csv(file_path, encoding="cp949")
            elif fn.endswith(".xlsx"):
                df = pd.read_excel(file_path)
            else:
                continue
        
            df = df.fillna("")

            # ✅ 파일 단위 정당 매핑 생성
            party_map = build_requester_party_map(df)

            # 발언자별로 발언 합치기
            grouped = (
                df.groupby("speaker_name")["speech_text"]
                .apply(lambda x: " ".join([t for t in x.astype(str).tolist() if t.strip()]))
                .reset_index()
                .rename(columns={"speech_text": "speeches"})
            )

            for _, row in tqdm(grouped.iterrows(), total=len(grouped)):
                requester = str(row["speaker_name"]).strip()
                speeches = str(row["speeches"]).strip()

                if not requester or not speeches:
                    continue

                # ✅ 1단계: 후보 필터
                if not is_candidate_request(speeches):
                    continue

                candidate_count += 1

                # ✅ 2단계: GPT 정밀 추출
                requests = extract_requests_from_speech(speeches, model=model)

                # ✅ 요구자 정당 매핑 ('미분류'는 공백)
                requester_party = get_requester_party(party_map, requester)

                for req in requests:
                    all_results.append({
                        "회의회차": meeting_round,
                        "요구자명": requester,
                        "요구자정당": requester_party,  # ✅ 추가
                        "대상": req.get("target", ""),
                        "실제요구자료": req.get("requested_data", ""),
                        "카테고리": req.get("category", "")
                    })

                time.sleep(sleep_sec)

    print("\n후보 발언 수:", candidate_count)
    print("추출 결과 수:", len(all_results))

    return pd.DataFrame(all_results)


# ===== Cell 10 =====
DATA_DIR = r"C:\Users\User\Desktop\data_collection\data"


# ===== Cell 11 =====
result_df = run_request_extraction_pipeline(
    data_dir=DATA_DIR,
    model="gpt-4.1-mini",
    sleep_sec=0.25
)


# ===== Cell 12 =====
output_path = os.path.join(DATA_DIR, "요구자료_추출결과_정당매핑.csv")
result_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print("저장 완료:", output_path)


# ===== Cell 13 =====
result_df.head()


후보 발언 수: 0
추출 결과 수: 0
저장 완료: C:\Users\User\Desktop\data_collection\data\요구자료_추출결과_정당매핑.csv


""


In [17]:
API_KEY = ''
client = OpenAI(
    api_key=API_KEY
)  # OPENAI_API_KEY 환경변수에 세팅되어 있어야 함
MODEL_NAME = "gpt-4.1-mini"  # 필요시 변경


In [18]:
# 예: C:\Users\User\Desktop\data_collection\data
DATA_DIR = r"C:\Users\User\Desktop\data_collection\data"


In [19]:
import re

REQUEST_KEYWORDS = [
    "자료", "제출", "현황", "통계", "보고서", "명단",
    "내역", "제공", "요구", "주시기 바랍니다", "바랍니다"
]

def is_candidate_request(text: str) -> bool:
    """
    자료 요구 가능성이 있는 발언인지 키워드로 1차 필터링
    """
    if not isinstance(text, str):
        return False
    return any(kw in text for kw in REQUEST_KEYWORDS)


In [20]:
SYSTEM_PROMPT = """
너는 대한민국 국회 회의록에서
‘실제 자료·문서·파일 제출 요구’만을 정밀하게 추출하는 전문가다.

━━━━━━━━━━━━━━━━━━━━━━
[자료 제출 요구의 엄격한 정의]
━━━━━━━━━━━━━━━━━━━━━━

다음 조건을 모두 만족하는 경우만 ‘자료 제출 요구’로 판단한다.

1. 요구 대상이 명확히 존재해야 한다.
   - 정부, 부처, 청, 위원회, 공공기관, 산하기관 등

2. 요구 내용이 ‘구체적인 자료의 제출’이어야 한다.
   포함 예시:
   - 통계, 현황표, 명단, 보고서, 문서, 자료집
   - 서면 자료, 파일, 내부 문건, 회의자료
   - 조사 결과를 정리한 보고서 형태의 자료

3. 단순 행동·조치·답변 요구는 제외한다.
   반드시 ‘자료를 제출하라 / 내라 / 제공하라’는 의미가 있어야 한다.

━━━━━━━━━━━━━━━━━━━━━━
[반드시 제외할 것]
━━━━━━━━━━━━━━━━━━━━━━

다음은 절대 포함하지 않는다.

- “조치하라”, “개선하라”, “검토하라” 등 행동 요구
- “설명해 달라”, “답변해 달라” 등 구두·서면 답변 요구
- 의견 표명, 문제 제기, 비판, 평가
- 자료 제출 없이 결과만 묻는 질문
- 이미 제출된 자료를 단순 언급하는 발언
- 과거 자료 요구를 회상만 하는 경우

━━━━━━━━━━━━━━━━━━━━━━
[요약 방식 — 매우 중요]
━━━━━━━━━━━━━━━━━━━━━━

‘requested_data’는 반드시 다음 형식을 따른다.

- 어떤 자료를
- 왜 요구하는지(문제의식·검증 목적·확인 목적 등)

을 **하나의 완결된 문장**으로 요약한다.

❌ 나쁜 예:
- “경찰 인력 현황 자료 제출 요구”

✅ 좋은 예:
- “지역별 경찰 인력 배치가 적정한지 검증하기 위해 최근 5년간 인력 현황 자료 제출 요구”

━━━━━━━━━━━━━━━━━━━━━━
[출력 형식]
━━━━━━━━━━━━━━━━━━━━━━

출력은 반드시 JSON 배열만 사용한다.
JSON 외의 다른 텍스트는 절대 출력하지 마라.

[
  {
    "target": "자료 제출을 요구받은 대상(기관/부처/조직)",
    "requested_data": "어떤 자료를 왜 요구하는지 한 문장 요약",
    "category": "아래 범주 중 하나"
  }
]

━━━━━━━━━━━━━━━━━━━━━━
[category 범주]
━━━━━━━━━━━━━━━━━━━━━━
- 범죄·치안
- 재난·안전
- 인사·조직
- 예산·재정
- 정책·제도
- 통계·현황
- 감사·점검
- 기타

━━━━━━━━━━━━━━━━━━━━━━
[출력 규칙]
━━━━━━━━━━━━━━━━━━━━━━

- 하나의 발언에서 여러 자료 제출 요구가 있으면 각각 분리한다.
- 자료 제출 요구가 전혀 없으면 반드시 빈 배열 [] 만 출력한다.
- 판단이 애매한 경우에는 포함하지 말고 제외한다.
"""


In [21]:
def build_user_prompt(speech_text: str) -> str:
    return f"""
다음은 국회 회의록 발언이다.
위 발언에서 '실제 자료 제출 요구'만 JSON 배열로 추출하라.

발언:
\"\"\"
{speech_text}
\"\"\"
"""


In [22]:
def extract_requests_from_speech(speech_text: str, model: str = "gpt-4.1-mini"):
    """
    GPT로 실제 자료 제출 요구를 JSON 배열로 추출.
    - quota/429 등 에러 시에도 파이프라인이 중단되지 않도록 [] 반환
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_user_prompt(speech_text)}
            ],
            temperature=0
        )
        content = response.choices[0].message.content.strip()
        return json.loads(content)

    except Exception as e:
        print("GPT 파싱 에러:", e)
        return []



In [23]:
REQUEST_KEYWORDS = [
    "자료", "제출", "현황", "통계", "보고서", "명단", "내역", "서면", "제공", "주시기 바랍니다", "바랍니다"
]

def is_candidate_request(text: str) -> bool:
    if not isinstance(text, str):
        return False
    t = text.strip()
    if not t:
        return False
    return any(k in t for k in REQUEST_KEYWORDS)

In [24]:
def build_requester_party_map(df: pd.DataFrame) -> dict:
    """
    ✅ 핵심: 요구자(의원/위원) 정당 매핑 생성
    - df 내 speaker_name별로 party를 매핑
    - party가 '미분류'면 공백으로 처리
    - 보통 요구자료는 의원/위원이 요구하므로,
      speaker_position에 '의원/위원/위원장/간사' 등이 포함된 경우에만 정당을 채움
      그 외(정부/기관/증인 등)는 공백으로 둠
    """
    required_cols = ["speaker_name", "speaker_position", "party"]
    for c in required_cols:
        if c not in df.columns:
            return {}

    tmp = df[required_cols].fillna("")
    tmp["speaker_name"] = tmp["speaker_name"].astype(str).str.strip()
    tmp["speaker_position"] = tmp["speaker_position"].astype(str).str.strip()
    tmp["party"] = tmp["party"].astype(str).str.strip()

    party_map = {}

    for name, g in tmp.groupby("speaker_name"):
        if not name:
            continue

        pos_join = " ".join(g["speaker_position"].tolist())
        is_lawmaker_like = any(k in pos_join for k in ["의원", "위원", "위원장", "간사"])

        parties = [p for p in g["party"].tolist() if p and p != "미분류"]

        if is_lawmaker_like and parties:
            party_map[name] = parties[0]
        else:
            party_map[name] = ""

    return party_map


def get_requester_party(party_map: dict, requester_name: str) -> str:
    return party_map.get(str(requester_name).strip(), "")


# ===== Cell 8 =====
def extract_meeting_round_from_folder(folder_name: str) -> str:
    # '제421회' -> '421회'
    return folder_name.replace("제", "").replace("회", "") + "회"


In [25]:
def run_request_extraction_pipeline(
    data_dir: str,
    file_suffix: str,
    model: str = "gpt-4.1-mini",
    sleep_sec: float = 0.25
) -> pd.DataFrame:
    """
    data_dir 하위의 '제379회'~ 폴더들을 순회하며,
    회차별로 발언자(speaker_name) 발언을 합쳐 '실제 자료 제출 요구'만 추출.

    추가 컬럼:
      - 요구자정당: df 내 party를 speaker_name 기준으로 매핑
                 (단, 의원/위원일 때만 채우고 '미분류'는 공백)

    출력 컬럼:
      회의회차, 요구자명, 요구자정당, 대상, 실제요구자료, 카테고리
    """
    all_results = []
    candidate_count = 0

    folders = [f for f in os.listdir(data_dir) if f.startswith("제") and f.endswith("회")]
    folders.sort(key=lambda x: int(x.replace("제", "").replace("회", "")))

    for folder in folders:
        meeting_round = extract_meeting_round_from_folder(folder)
        folder_path = os.path.join(data_dir, folder)
        print(f"\n▶ 처리 중: {meeting_round}")

        files = [fn for fn in os.listdir(folder_path) if fn.endswith(file_suffix)]
        files.sort()

        for fn in files:
            file_path = os.path.join(folder_path, fn)
            df = pd.read_csv(file_path).fillna("")

            # ✅ 파일 단위 정당 매핑 생성
            party_map = build_requester_party_map(df)

            # 발언자별로 발언 합치기
            grouped = (
                df.groupby("speaker_name")["speech_text"]
                .apply(lambda x: " ".join([t for t in x.astype(str).tolist() if t.strip()]))
                .reset_index()
                .rename(columns={"speech_text": "speeches"})
            )

            for _, row in tqdm(grouped.iterrows(), total=len(grouped)):
                requester = str(row["speaker_name"]).strip()
                speeches = str(row["speeches"]).strip()

                if not requester or not speeches:
                    continue

                # ✅ 1단계: 후보 필터
                if not is_candidate_request(speeches):
                    continue

                candidate_count += 1

                # ✅ 2단계: GPT 정밀 추출
                requests = extract_requests_from_speech(speeches, model=model)

                # ✅ 요구자 정당 매핑 ('미분류'는 공백)
                requester_party = get_requester_party(party_map, requester)

                for req in requests:
                    all_results.append({
                        "회의회차": meeting_round,
                        "요구자명": requester,
                        "요구자정당": requester_party,  # ✅ 추가
                        "대상": req.get("target", ""),
                        "실제요구자료": req.get("requested_data", ""),
                        "카테고리": req.get("category", "")
                    })

                time.sleep(sleep_sec)

    print("\n후보 발언 수:", candidate_count)
    print("추출 결과 수:", len(all_results))

    return pd.DataFrame(all_results)


In [26]:
output_path = os.path.join(DATA_DIR, "요구자료_추출결과_정당매핑.csv")
result_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print("저장 완료:", output_path)


저장 완료: C:\Users\User\Desktop\data_collection\data\요구자료_추출결과_정당매핑.csv


In [27]:
result_df.head()

""


In [28]:
result_df = pd.DataFrame(all_results)

output_path = os.path.join(DATA_DIR, "요구자료_목록_379_414회_통합.csv")
result_df.to_csv(output_path, index=False, encoding="utf-8-sig")

output_path


NameError: name 'all_results' is not defined

In [ ]:

# CSV 파일 경로
file_path = r"C:\\Users\\User\\Desktop\\data_collection\\data\\요구자료_목록_415_430회_통합.csv"

# CSV 불러오기
df = pd.read_csv(file_path)

# 전체 행 개수 출력
print("전체 행 개수:", len(df))


In [15]:
df = pd.read_csv(file_path)

# ✅ 422회차만 필터링
df_422 = df[df["회의회차"] == "427회"]

# 결과 확인
print("422회차 요구자료 개수:", len(df_422))
df_422.head()

NameError: name 'file_path' is not defined